# Car Price Prediction Pipeline
## From Raw Craigslist Scrapes to Champion Models on GCP

**OPIM 5512 - Dr. Dave Wanik - University of Connecticut**

---

This notebook walks through the full data pipeline:

1. **Extract** structured fields from raw Craigslist `.txt` scrape files (regex)
2. **Merge** LLM-extracted CSV files into a clean baseline
3. **Train** two champion models -- one regex-based, one LLM-based
4. **Evaluate** with train/test split, cross-validation, and feature importance
5. **Save** models as `.pkl` files
6. **Upload** models and baselines to Google Cloud Storage

The trained models become the **champions** in a live champion-vs-challenger
system running hourly on GCP Cloud Functions.

> **Data:** 203,704 raw scrape files from Craigslist New Haven (Nov 2025 - May 2026)
> stored in `scrapes_backup.zip`

## 0. Setup

In [ ]:
import csv
import glob
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

# -- paths --
DOWNLOADS   = r"C:\Users\dww05002\Downloads"
ZIP_PATH    = rf"{DOWNLOADS}\scrapes_backup.zip"
MODELS_DIR  = rf"{DOWNLOADS}\models"
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

GCS_BUCKET  = "craigslist-scraper-bucket"
GCS_PROJECT = "craigslist-scraper-499015"
print("Setup complete.")

## 1. Peek at the Raw Data

Each run folder contains one `.txt` file per listing scraped from Craigslist.
The files are raw HTML-stripped text -- messy but consistent.

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    all_names = z.namelist()
    txt_files = [n for n in all_names if n.endswith(".txt")]
    run_folders = sorted({n.split("/")[1] for n in all_names if len(n.split("/")) >= 2 and n.split("/")[1]})

print(f"Total .txt files : {len(txt_files):,}")
print(f"Unique run folders: {len(run_folders):,}")
print(f"First run : {run_folders[0]}")
print(f"Last run  : {run_folders[-1]}")
print(f"\nSame listing scraped many times -- dedup gives us unique cars")
# peek at one raw file
with zipfile.ZipFile(ZIP_PATH) as z:
    sample = z.read(txt_files[0]).decode("utf-8", errors="replace")

print("\n--- Sample raw listing (first 600 chars) ---")
print(sample[:600])

## 2. Regex Extraction

We parse structured fields from each raw `.txt` file using regular expressions.
The key insight: Craigslist uses a consistent layout, so the price, year, make,
model, mileage, transmission, and color are always in the same spots.

**Why regex first?**
- Fast: processes 200k files in ~3 minutes
- Free: no API calls
- Good enough: 97%+ hit rate on price, year, make

We deduplicate by `post_id` -- keeping the first time we saw each listing.

In [ ]:
# regex patterns derived from inspecting raw files
RE_PRICE        = re.compile(r"\n\$([\d,]+)\n")
RE_YEAR_MAKE    = re.compile(r"^\s*(\d{4})\s*\n\s*([a-z0-9 &\-/.]+)\n", re.IGNORECASE | re.MULTILINE)
RE_ODOMETER     = re.compile(r"odometer:\s*\n?\s*([^\n]+)", re.IGNORECASE)
RE_TRANSMISSION = re.compile(r"transmission:\s*\n?\s*([^\n]+)", re.IGNORECASE)
RE_COLOR        = re.compile(r"paint color:\s*\n?\s*([^\n]+)", re.IGNORECASE)
RE_CONDITION    = re.compile(r"condition:\s*\n?\s*([^\n]+)", re.IGNORECASE)
RE_FUEL         = re.compile(r"fuel:\s*\n?\s*([^\n]+)", re.IGNORECASE)
RE_TYPE         = re.compile(r"type:\s*\n?\s*([^\n]+)", re.IGNORECASE)
RE_POST_ID      = re.compile(r"post id:\s*(\d+)", re.IGNORECASE)
RE_POSTED       = re.compile(r"Posted\s*\n\s*(\d{4}-\d{2}-\d{2})", re.IGNORECASE)

def parse_txt(text, run_id, post_id_path):
    def _num(s): return re.sub(r"[^\d]", "", s or "")
    def _m(pattern): m = pattern.search(text); return m.group(1).strip() if m else ""

    pid = _m(RE_POST_ID) or post_id_path
    m_ym = RE_YEAR_MAKE.search(text)
    year = m_ym.group(1) if m_ym else ""
    make_model = m_ym.group(2).strip().lower() if m_ym else ""
    parts = make_model.split(None, 1)

    return {
        "post_id":      pid,
        "run_id":       run_id,
        "scraped_at":   _m(RE_POSTED),
        "price":        _num(_m(RE_PRICE)),
        "year":         year,
        "make":         parts[0].title() if parts else "",
        "model":        parts[1].title() if len(parts) > 1 else "",
        "mileage":      _num(_m(RE_ODOMETER)),
        "transmission": _m(RE_TRANSMISSION).lower(),
        "color":        _m(RE_COLOR).lower(),
        "condition":    _m(RE_CONDITION).lower(),
        "fuel":         _m(RE_FUEL).lower(),
        "vehicle_type": _m(RE_TYPE).lower(),
        "source_txt":   text[:300].replace("\n", " "),
    }

print("Parsing functions defined.")

In [ ]:
# run extraction on all 203k files
print(f"Extracting from {len(txt_files):,} files...")
seen, records, errors = set(), [], 0

with zipfile.ZipFile(ZIP_PATH) as z:
    for i, name in enumerate(txt_files):
        if i % 25000 == 0:
            print(f"  {i:>7,} / {len(txt_files):,}  ({len(records):,} unique so far)")
        parts = name.split("/")
        run_id = parts[1] if len(parts) >= 2 else ""
        pid_path = Path(parts[-1]).stem
        try:
            raw = z.read(name).decode("utf-8", errors="replace")
            rec = parse_txt(raw, run_id, pid_path)
        except Exception:
            errors += 1
            continue
        pid = rec["post_id"]
        if pid and pid not in seen:
            seen.add(pid)
            records.append(rec)

df_raw = pd.DataFrame(records)
print(f"\nExtracted {len(df_raw):,} unique listings ({errors} errors)")
df_raw.head(3)

In [ ]:
# clean: require valid price, year, make
df_raw["price_n"]   = pd.to_numeric(df_raw["price"], errors="coerce")
df_raw["year_n"]    = pd.to_numeric(df_raw["year"],  errors="coerce")
df_raw["mileage_n"] = pd.to_numeric(df_raw["mileage"], errors="coerce")

df_regex = df_raw[
    df_raw["price_n"].between(500, 150_000) &
    df_raw["year_n"].between(1980, 2027) &
    df_raw["make"].notna() & (df_raw["make"] != "")
].copy()

print(f"Clean rows: {len(df_regex):,}  (dropped {len(df_raw)-len(df_regex):,})")
print(f"Date range: {df_regex.scraped_at.min()} --> {df_regex.scraped_at.max()}")
print(f"\nField completeness:")
for col in ["price","year","make","model","mileage","transmission","color"]:
    pct = (df_regex[col] != "").mean() * 100
    print(f"  {col:<14} {pct:.1f}%")

# save
OUT_REGEX = rf"{DOWNLOADS}\baseline_regex_clean.csv"
df_regex.to_csv(OUT_REGEX, index=False)
print(f"\nSaved: {OUT_REGEX}")

## 3. Merge LLM-Extracted CSVs

The LLM extractor (Vertex AI / Gemini) ran on a subset of listings and produced
richer structured output. We merge all LLM CSV snapshots into one deduplicated
baseline, preferring rows with more fields filled in (LLM v1 has `color` too).

In [ ]:
llm_files = sorted(glob.glob(rf"{DOWNLOADS}\structured_datasets_listings_master_llm*.csv"))
print(f"Found {len(llm_files)} LLM files:")
for f in llm_files:
    df = pd.read_csv(f)
    print(f"  {len(df):>4} rows  {Path(f).name}")

frames = [pd.read_csv(f) for f in llm_files]
combined = pd.concat(frames, ignore_index=True)
print(f"\nCombined: {len(combined):,} rows before dedup")

In [ ]:
# dedup by post_id -- keep row with most fields filled (prefer LLM v1)
combined["_filled"] = combined.notna().sum(axis=1)
combined = (combined.sort_values("_filled", ascending=False)
                    .drop_duplicates(subset="post_id", keep="first")
                    .drop(columns=["_filled"])
                    .sort_values("scraped_at").reset_index(drop=True))

# clean
combined["price_n"] = pd.to_numeric(
    combined["price"].astype(str).str.replace(r"[^\d]","",regex=True), errors="coerce")
combined["year_n"]  = pd.to_numeric(combined["year"], errors="coerce")

df_llm = combined[
    combined["price_n"].between(500, 150_000) &
    combined["year_n"].between(1980, 2027) &
    combined["make"].notna()
].copy()

print(f"Unique LLM listings: {len(df_llm):,}")
print(f"Date range: {df_llm.scraped_at.min()} --> {df_llm.scraped_at.max()}")
print(f"\nField completeness:")
for col in ["price","year","make","model","mileage","transmission","color"]:
    if col in df_llm.columns:
        pct = df_llm[col].notna().mean() * 100
        print(f"  {col:<14} {pct:.1f}%")

OUT_LLM = rf"{DOWNLOADS}\baseline_llm_clean.csv"
df_llm.to_csv(OUT_LLM, index=False)
print(f"\nSaved: {OUT_LLM}")

## 4. Exploratory Data Analysis

Before training, understand the data. What does the price distribution look like?
Which makes appear most often? Does year correlate strongly with price?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Regex Baseline: 4,185 Craigslist Car Listings", fontsize=14, fontweight="bold")

# Price distribution
ax = axes[0,0]
df_regex["price_n"].clip(upper=40000).hist(bins=50, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Price Distribution (capped at $40k)")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_xlabel("Price"); ax.set_ylabel("Count")

# Top makes
ax = axes[0,1]
top_makes = df_regex["make"].value_counts().head(12)
top_makes.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 12 Makes")
ax.set_xlabel("Listings")
ax.invert_yaxis()

# Year vs price
ax = axes[1,0]
sample = df_regex.sample(min(1000, len(df_regex)), random_state=42)
ax.scatter(sample["year_n"], sample["price_n"].clip(upper=40000),
           alpha=0.3, s=10, color="steelblue")
ax.set_title("Year vs Price")
ax.set_xlabel("Year"); ax.set_ylabel("Price ($)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Transmission breakdown
ax = axes[1,1]
trans = df_regex["transmission"].replace("", "unknown").value_counts()
trans.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Transmission Types")
ax.set_xlabel(""); ax.set_ylabel("Count")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

## 5. Train Champion Models

We train two champions using a **sklearn Pipeline**:
- `ColumnTransformer` handles numeric imputation + one-hot encoding of categoricals
- `DecisionTreeRegressor` is the model (interpretable, no feature scaling needed)

Each champion is evaluated with:
- Train / test split (80/20)
- 5-fold cross-validation MAE
- Feature importance ranking

In [ ]:
REGEX_CAT = ["make","model","transmission","color","fuel","vehicle_type"]
LLM_CAT   = ["make","model","transmission","color"]
NUM_COLS  = ["year_n","mileage_n"]
TARGET    = "price_n"
RANDOM_STATE = 42

def build_pipeline(cat_cols):
    pre = ColumnTransformer(transformers=[
        ("num", SimpleImputer(strategy="median"), NUM_COLS),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ], remainder="drop")
    return Pipeline([("pre", pre),
                     ("model", DecisionTreeRegressor(
                         max_depth=12, min_samples_leaf=10, random_state=RANDOM_STATE))])

def prep(df, cat_cols):
    df = df.copy()
    for c in cat_cols:
        if c not in df.columns: df[c] = np.nan
    return df

print("Pipeline builder defined.")

In [ ]:
# -- Champion Regex --
print("Training champion_regex...")
df_r = prep(df_regex, REGEX_CAT)
X_r, y_r = df_r[REGEX_CAT + NUM_COLS], df_r[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=RANDOM_STATE)

pipe_regex = build_pipeline(REGEX_CAT)
pipe_regex.fit(X_tr, y_tr)

train_mae_r = mean_absolute_error(y_tr, pipe_regex.predict(X_tr))
test_mae_r  = mean_absolute_error(y_te, pipe_regex.predict(X_te))
test_r2_r   = r2_score(y_te, pipe_regex.predict(X_te))
cv_r        = -cross_val_score(pipe_regex, X_r, y_r, cv=5, scoring="neg_mean_absolute_error")

print(f"  Train rows : {len(X_tr):,}")
print(f"  Train MAE  : ${train_mae_r:,.0f}")
print(f"  Test MAE   : ${test_mae_r:,.0f}  (R2={test_r2_r:.3f})")
print(f"  5-fold MAE : ${cv_r.mean():,.0f} +/- ${cv_r.std():,.0f}")

In [ ]:
# -- Champion LLM --
print("Training champion_llm...")
df_l = prep(df_llm, LLM_CAT)
X_l, y_l = df_l[LLM_CAT + NUM_COLS], df_l[TARGET]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_l, y_l, test_size=0.2, random_state=RANDOM_STATE)

pipe_llm = build_pipeline(LLM_CAT)
pipe_llm.fit(X_tr2, y_tr2)

train_mae_l = mean_absolute_error(y_tr2, pipe_llm.predict(X_tr2))
test_mae_l  = mean_absolute_error(y_te2, pipe_llm.predict(X_te2))
test_r2_l   = r2_score(y_te2, pipe_llm.predict(X_te2))
cv_l        = -cross_val_score(pipe_llm, X_l, y_l, cv=5, scoring="neg_mean_absolute_error")

print(f"  Train rows : {len(X_tr2):,}")
print(f"  Train MAE  : ${train_mae_l:,.0f}")
print(f"  Test MAE   : ${test_mae_l:,.0f}  (R2={test_r2_l:.3f})")
print(f"  5-fold MAE : ${cv_l.mean():,.0f} +/- ${cv_l.std():,.0f}")

## 6. Model Comparison & Feature Importance

Which champion performs better? What features matter most?

**Key insight:** Year and mileage dominate both models. Make/model/color add
small but meaningful signal. The LLM champion has lower MAE but was trained on
a narrower price range -- it's not directly comparable to the regex champion.

In [ ]:
# side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pipe, cat_cols, label, cv, test_mae, test_r2 in [
    (axes[0], pipe_regex, REGEX_CAT, "Champion Regex",  cv_r, test_mae_r, test_r2_r),
    (axes[1], pipe_llm,   LLM_CAT,   "Champion LLM",    cv_l, test_mae_l, test_r2_l),
]:
    ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["oh"]
    feat_names = NUM_COLS + list(ohe.get_feature_names_out())
    importances = pipe.named_steps["model"].feature_importances_
    top = sorted(zip(feat_names, importances), key=lambda x: x[1], reverse=True)[:12]
    names, vals = zip(*top)

    ax.barh(range(len(names)), vals, color="steelblue")
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels([n.replace("x0_","make:").replace("x1_","model:")
                          .replace("x2_","trans:").replace("x3_","color:")
                          .replace("x4_","fuel:").replace("x5_","type:") for n in names],
                       fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"{label}\nTest MAE=${test_mae:,.0f}  R2={test_r2:.3f}  CV MAE=${cv.mean():,.0f}")
    ax.set_xlabel("Feature Importance")

plt.suptitle("Feature Importance: What Drives Car Prices?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# actual vs predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pipe, X_te, y_te, label in [
    (axes[0], pipe_regex, X_te,  y_te,  "Champion Regex"),
    (axes[1], pipe_llm,   X_te2, y_te2, "Champion LLM"),
]:
    y_hat = pipe.predict(X_te)
    mae = mean_absolute_error(y_te, y_hat)
    ax.scatter(y_te.clip(upper=40000), np.clip(y_hat, 0, 40000),
               alpha=0.3, s=10, color="steelblue")
    lims = [0, 40000]
    ax.plot(lims, lims, "r--", linewidth=1, label="Perfect")
    ax.set_title(f"{label}  MAE=${mae:,.0f}")
    ax.set_xlabel("Actual Price ($)")
    ax.set_ylabel("Predicted Price ($)")
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x/1000:.0f}k"))
    ax.legend(fontsize=8)

plt.suptitle("Actual vs Predicted (capped at $40k)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Save Models as .pkl

We serialize both trained pipelines using `joblib`. The entire sklearn Pipeline
(preprocessor + model) is saved, so we can call `.predict()` on raw data later
without re-fitting transformers.

In [ ]:
REGEX_PKL = rf"{MODELS_DIR}\champion_regex.pkl"
LLM_PKL   = rf"{MODELS_DIR}\champion_llm.pkl"

joblib.dump(pipe_regex, REGEX_PKL)
joblib.dump(pipe_llm,   LLM_PKL)

print(f"Saved: {REGEX_PKL}")
print(f"Saved: {LLM_PKL}")

# verify: reload and score
pipe_test = joblib.load(REGEX_PKL)
reloaded_mae = mean_absolute_error(y_te, pipe_test.predict(X_te))
print(f"\nVerification -- reloaded model test MAE: ${reloaded_mae:,.0f}  (should match above)")

In [ ]:
# save metrics for the Cloud Function / comparison notebook
metrics = [
    {
        "champion":      "champion_regex",
        "trained_at":    datetime.now(timezone.utc).isoformat(),
        "train_rows":    int(len(X_tr)),
        "test_rows":     int(len(X_te)),
        "train_mae":     round(float(train_mae_r), 2),
        "test_mae":      round(float(test_mae_r),  2),
        "test_r2":       round(float(test_r2_r),   4),
        "cv_mae_mean":   round(float(cv_r.mean()), 2),
        "cv_mae_std":    round(float(cv_r.std()),  2),
        "price_median":  round(float(y_r.median()), 2),
    },
    {
        "champion":      "champion_llm",
        "trained_at":    datetime.now(timezone.utc).isoformat(),
        "train_rows":    int(len(X_tr2)),
        "test_rows":     int(len(X_te2)),
        "train_mae":     round(float(train_mae_l), 2),
        "test_mae":      round(float(test_mae_l),  2),
        "test_r2":       round(float(test_r2_l),   4),
        "cv_mae_mean":   round(float(cv_l.mean()), 2),
        "cv_mae_std":    round(float(cv_l.std()),  2),
        "price_median":  round(float(y_l.median()), 2),
    },
]

METRICS_JSON = rf"{MODELS_DIR}\champion_metrics.json"
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved: {METRICS_JSON}")
print(json.dumps(metrics, indent=2))

## 8. Upload to Google Cloud Storage

The `.pkl` files and baseline CSVs are uploaded to GCS so the Cloud Function
can load them at runtime without any local files.

**GCS layout after upload:**
```
gs://craigslist-scraper-bucket/
  models/
    champion_regex.pkl
    champion_llm.pkl
    champion_metrics.json
  structured/datasets/
    baseline_regex_clean.csv
    baseline_llm_clean.csv
```

> **Note:** Requires `google-cloud-storage` and Application Default Credentials.
> Run `gcloud auth application-default login` once before this cell,
> or upload manually via Cloud Shell (see GCP_SETUP.md).

In [ ]:
from google.cloud import storage as gcs

def upload(local_path, gcs_key, bucket_name=GCS_BUCKET, project=GCS_PROJECT):
    client = gcs.Client(project=project)
    blob   = client.bucket(bucket_name).blob(gcs_key)
    blob.upload_from_filename(local_path)
    print(f"  uploaded: gs://{bucket_name}/{gcs_key}")

print("Uploading models...")
upload(REGEX_PKL,    "models/champion_regex.pkl")
upload(LLM_PKL,      "models/champion_llm.pkl")
upload(METRICS_JSON, "models/champion_metrics.json")

print("\nUploading baselines...")
upload(OUT_REGEX, "structured/datasets/baseline_regex_clean.csv")
upload(OUT_LLM,   "structured/datasets/baseline_llm_clean.csv")

print("\nAll uploads complete.")

## 9. What Happens Next (Live Pipeline)

Once the models are in GCS, the **train-dt** Cloud Function runs hourly and:

1. Loads `champion_regex.pkl` and `champion_llm.pkl` from GCS
2. Retrains two **challengers** on the latest scraped data
3. Scores today's holdout listings against all 4 models
4. Writes `comparison.csv` to GCS with side-by-side predictions

**Champion vs Challenger logic:**
- Champions are frozen -- they never change unless YOU retrain and re-upload
- Challengers see new data every hour -- they may get better or worse over time
- When challenger MAE consistently beats champion, you promote it to champion

This is how production ML systems work in the real world.

In [ ]:
# summary table
summary = pd.DataFrame(metrics)[
    ["champion","train_rows","test_mae","test_r2","cv_mae_mean","price_median"]
].rename(columns={
    "champion":     "Model",
    "train_rows":   "Train Rows",
    "test_mae":     "Test MAE ($)",
    "test_r2":      "R2",
    "cv_mae_mean":  "CV MAE ($)",
    "price_median": "Median Price ($)",
})
summary["Test MAE ($)"]     = summary["Test MAE ($)"].apply(lambda x: f"${x:,.0f}")
summary["CV MAE ($)"]       = summary["CV MAE ($)"].apply(lambda x: f"${x:,.0f}")
summary["Median Price ($)"] = summary["Median Price ($)"].apply(lambda x: f"${x:,.0f}")
print(summary.to_string(index=False))
print("\nBoth champions saved to GCS and ready for the live pipeline.")